# Chapter 15: Recursion
### COSC 221 — Computer Science B

**Lecture Notes**

## Objectives
- Understand the concept of recursion
- Learn to think recursively
- Compare recursion with iteration
- Solve problems using recursion
- Understand the pros and cons of recursion
- Apply recursion to classic algorithmic challenges

## Contents
1. Introduction (Factorial Function)
2. Base Case and Recursive Case
3. Fibonacci Numbers
4. Palindrome Problem
5. Selection Sort
6. Binary Search
7. Finding Directory Size
8. Recursion vs Iteration
9. Advantages and Disadvantages of Recursion


---
## 1. Introduction: The Factorial Function

**Recursion** is a technique where a function solves a problem by calling *itself* on a smaller version of the same problem.

A classic first example is the **factorial** function:

$$n! = n \times (n-1) \times (n-2) \times \dots \times 1, \qquad 0! = 1$$

Notice that this definition is naturally recursive:

$$n! = n \times (n-1)!$$

with the stopping point $0! = 1$.

This gives us the two ingredients every recursive function needs:
- A **base case** — the simplest version of the problem, solved directly (here, $0! = 1$)
- A **recursive case** — the function calls itself on a smaller sub-problem (here, $n \times (n-1)!$)


In [1]:
def factorial(n):
    if n == 0:          # base case
        return 1
    else:                # recursive case
        return n * factorial(n - 1)

# quick check
for i in range(6):
    print(f"{i}! = {factorial(i)}")


0! = 1
1! = 1
2! = 2
3! = 6
4! = 24
5! = 120


### Tracing the call stack

Calling `factorial(4)` unwinds like this:

```
factorial(4)
= 4 * factorial(3)
= 4 * (3 * factorial(2))
= 4 * (3 * (2 * factorial(1)))
= 4 * (3 * (2 * (1 * factorial(0))))
= 4 * (3 * (2 * (1 * 1)))
= 24
```

Each call is placed on the **call stack** and waits for the recursive call below it to return before it can compute its own result.


In [2]:
def factorial_traced(n, depth=0):
    indent = "  " * depth
    print(f"{indent}factorial({n}) called")
    if n == 0:
        print(f"{indent}-> base case reached, returning 1")
        return 1
    result = n * factorial_traced(n - 1, depth + 1)
    print(f"{indent}-> factorial({n}) returns {result}")
    return result

factorial_traced(4)


factorial(4) called
  factorial(3) called
    factorial(2) called
      factorial(1) called
        factorial(0) called
        -> base case reached, returning 1
      -> factorial(1) returns 1
    -> factorial(2) returns 2
  -> factorial(3) returns 6
-> factorial(4) returns 24


24

---
## 2. Base Case and Recursive Case

**Key ideas:**

- Recursion is often used together with `if`/`else` statements that distinguish different cases.
- There must be **at least one base case** — a condition that stops the recursion.
- **Every recursive call must reduce the problem**, moving it closer to the base case.
- Many recursive problems can be expressed as a **recurrence relation** (a mathematical formula defining a sequence in terms of earlier terms).

If you forget the base case, or if the recursive call doesn't shrink the problem, the function will call itself forever — Python will eventually raise a `RecursionError` (stack overflow).


In [3]:
def no_base_case(n):
    # BAD EXAMPLE - do not run for large n, this will crash
    return n * no_base_case(n - 1)

try:
    no_base_case(5)
except RecursionError as e:
    print("RecursionError:", e)


RecursionError: maximum recursion depth exceeded


### General template for a recursive function

```python
def recursive_function(problem):
    if is_base_case(problem):
        return base_case_answer
    else:
        smaller_problem = shrink(problem)
        return combine(problem, recursive_function(smaller_problem))
```


---
## 3. Fibonacci Numbers

The Fibonacci sequence is defined by the recurrence:

$$F(0) = 0, \quad F(1) = 1, \quad F(n) = F(n-1) + F(n-2) \text{ for } n \ge 2$$

This has **two base cases** and a recursive case that calls itself **twice**.


In [4]:
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

print([fibonacci(i) for i in range(10)])


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


> **Note (efficiency):** Recursion is *not* an efficient way to compute Fibonacci numbers — plain recursive Fibonacci recomputes the same sub-problems many times (exponential time). It is, however, a great way to *understand* how recursion works. (We'll compare recursion and iteration in Section 8.)

### A lighter example: recursive thinking is everywhere

Thinking recursively can describe simple everyday actions too — for example, drinking water from a bottle one sip at a time:


In [5]:
class Bottle:
    def __init__(self, sips_left):
        self.sips_left = sips_left

    def isEmpty(self):
        return self.sips_left <= 0

    def takeOneSip(self):
        self.sips_left -= 1


def drinkWater(bottle):
    if not bottle.isEmpty():
        bottle.takeOneSip()          # take a sip, water decreases a bit
        print("sip! sips left:", bottle.sips_left)
        drinkWater(bottle)           # recursive call on the smaller problem


drinkWater(Bottle(5))


sip! sips left: 4
sip! sips left: 3
sip! sips left: 2
sip! sips left: 1
sip! sips left: 0


---
## 4. Palindrome Problem

A string is a **palindrome** if it reads the same from the left and from the right.

- `"level"` is a palindrome
- `"hello"` is not

**Challenge:** write a recursive function `isPalindrome(s)` to check whether a string `s` is a palindrome.

### 1. Palindrome problem with recursion

The recursive idea:
- **Base case:** a string of length 0 or 1 is always a palindrome.
- **Recursive case:** a string is a palindrome if its first and last characters match **and** the substring with those two characters removed is also a palindrome.


In [6]:
def isPalindrome(s):
    if len(s) <= 1:                       # base case
        return True
    if s[0] != s[-1]:                     # first mismatch -> not a palindrome
        return False
    return isPalindrome(s[1:-1])          # recursive case: shrink the string

for word in ["level", "hello", "Anna", "racecar", "python"]:
    print(word, "->", isPalindrome(word))


level -> True
hello -> False
Anna -> False
racecar -> True
python -> False


### 2. Recursive helper functions

The version above is **not efficient**: every recursive call creates a *new string* with slicing (`s[1:-1]`), which costs time and memory.

To avoid this, we can keep the original string fixed and instead use a **recursive helper function** that tracks index positions (`left` and `right`) into the *same* string.


In [7]:
def isPalindrome_helper(s):
    def helper(left, right):
        if left >= right:                 # base case: indices have met/crossed
            return True
        if s[left] != s[right]:
            return False
        return helper(left + 1, right - 1)  # recursive case: move indices inward

    return helper(0, len(s) - 1)

for word in ["level", "hello", "Anna", "racecar", "python"]:
    print(word, "->", isPalindrome_helper(word))


level -> True
hello -> False
Anna -> False
racecar -> True
python -> False


---
## 5. Selection Sort

Selection sort is based on two ideas:
1. Find the **smallest element** of a list, then swap it with the first element of the list.
2. **Ignore the first element** and continue sorting the (now shorter) remaining list.

### Example

Sort `L = [3, 2, 5, 1, 2, 6, 3]`

- Find the smallest element = `1`, swap with `3` → `L = [1, 2, 5, 3, 2, 6, 3]`
- Do the same with the smaller list `L1 = [2, 5, 3, 2, 6, 3]`
- Repeat: `L2 = [5, 3, 2, 6, 3] -> [2, 3, 5, 6, 3]`
- `L3 = [3, 5, 6, 3] -> L4 = [5, 6, 3] -> [3, 6, 5] -> L5 = [6, 5] -> [5, 6]`

This "shrink the remaining unsorted list each call" pattern is exactly the recursive structure.


In [8]:
def selection_sort(L):
    if len(L) <= 1:                       # base case: 0 or 1 elements is already sorted
        return L

    # find index of the smallest element
    min_index = 0
    for i in range(1, len(L)):
        if L[i] < L[min_index]:
            min_index = i

    # swap smallest element to the front
    L[0], L[min_index] = L[min_index], L[0]

    # recursively sort the rest of the list (everything after index 0)
    return [L[0]] + selection_sort(L[1:])


print(selection_sort([3, 2, 5, 1, 2, 6, 3]))


[1, 2, 2, 3, 3, 5, 6]


In [9]:
import random

# Generate a few random lists, and test out the function
random.seed(0)
for _ in range(3):
    test_list = [random.randint(0, 20) for _ in range(8)]
    print("before:", test_list, "-> after:", selection_sort(test_list.copy()))


before: [12, 13, 1, 8, 16, 15, 12, 9] -> after: [1, 8, 9, 12, 12, 13, 15, 16]
before: [15, 11, 18, 6, 16, 4, 9, 4] -> after: [4, 4, 6, 9, 11, 15, 16, 18]
before: [3, 19, 8, 17, 19, 4, 9, 3] -> after: [3, 3, 4, 8, 9, 17, 19, 19]


---
## 6. Binary Search

Just like searching for a letter in a pile of letters ordered by name, we often start in the **middle** and then move up or down depending on what we find.

**Binary search** looks for a key in a *sorted* list by repeatedly breaking it in half and continuing the search in the half where the key must lie.

### Example

Search for `7` in a sorted list `L` of size 10:
- Break `L` into `L1` and `L2` where `L2[0] > 7`
- Then look for `7` only in `L1`
- Repeat until found or the remaining list is empty

Requires the list to already be **sorted**.


In [10]:
def binary_search(L, key, low=0, high=None):
    if high is None:
        high = len(L) - 1

    if low > high:                        # base case: search space is empty
        return -1

    mid = (low + high) // 2
    if L[mid] == key:
        return mid                        # base case: found it
    elif L[mid] > key:
        return binary_search(L, key, low, mid - 1)   # search left half
    else:
        return binary_search(L, key, mid + 1, high)  # search right half


sorted_list = [1, 3, 4, 6, 7, 9, 12, 15, 20, 21]
for key in [7, 20, 5]:
    print(f"searching for {key}: index = {binary_search(sorted_list, key)}")


searching for 7: index = 4
searching for 20: index = 8
searching for 5: index = -1


In [11]:
import random

# Generate a few random lists, sort them, then search for a key in each
random.seed(1)
for _ in range(3):
    L = sorted(random.sample(range(50), 10))
    key = random.choice(L)
    print(f"list={L}, key={key}, found at index {binary_search(L, key)}")


list=[4, 7, 8, 16, 24, 28, 30, 31, 36, 41], key=16, found at index 3
list=[0, 1, 6, 14, 17, 24, 27, 28, 31, 38], key=38, found at index 9
list=[0, 1, 6, 13, 20, 24, 34, 41, 46, 47], key=34, found at index 6


---
## 7. Finding Directory Size

The size of a directory is the **sum of all file sizes** within that directory (including files inside its subdirectories).

Finding directory size is hard *without* recursion, because a directory can contain other directories, which contain other directories, and so on — an unknown depth of nesting.

To solve this recursively, we need three building blocks from the `os` module:
- `os.path.isfile(s)` — returns `True` if `s` is a filename
- `os.path.getsize(filename)` — returns the size of the file
- `os.listdir(directory)` — returns a list of the subdirectories and files under `directory`


In [12]:
import os

def getDirectorySize(directory):
    total = 0
    if os.path.isfile(directory):
        # base case: it's just a single file
        return os.path.getsize(directory)

    try:
        for item in os.listdir(directory):
            full_path = os.path.join(directory, item)
            total += getDirectorySize(full_path)   # recursive case
    except (FileNotFoundError, NotADirectoryError, PermissionError):
        return 0

    return total


In [13]:
# Test the function with a real directory, and a non-existing one

print("Size of current directory (bytes):", getDirectorySize("."))
print("Size of a non-existing directory:", getDirectorySize("no_such_directory_xyz"))


Size of current directory (bytes): 84255
Size of a non-existing directory: 0


---
## 8. Recursion vs Iteration

- Iteration often requires an explicit loop body and a known number of iterations, while recursion does not.
- Recursion generally consumes **more time and memory** than iteration (every call adds a frame to the call stack).
- Any task that can be solved recursively can generally also be solved with iteration — so why use recursion at all?
  - It can **simplify** the task and be **easier to code**, as in the directory-size problem above.
- Which method to use depends on the **nature of the problem**.
- Iteration is often **more efficient** than recursion when the task can be easily expressed with a loop.

### Example: Fibonacci, recursive vs iterative


In [14]:
import time

def fib_recursive(n):
    if n < 2:
        return n
    return fib_recursive(n - 1) + fib_recursive(n - 2)

def fib_iterative(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

n = 25
start = time.time()
r1 = fib_recursive(n)
t1 = time.time() - start

start = time.time()
r2 = fib_iterative(n)
t2 = time.time() - start

print(f"recursive fib({n}) = {r1}, time = {t1:.5f}s")
print(f"iterative fib({n}) = {r2}, time = {t2:.5f}s")


recursive fib(25) = 75025, time = 0.01066s
iterative fib(25) = 75025, time = 0.00004s


Even for a modest `n`, plain recursive Fibonacci is dramatically slower than the iterative version, because it repeats the same sub-calculations many times. This is a good illustration of why the *choice* between recursion and iteration matters in practice.


---
## 9. Advantages and Disadvantages of Recursion

### Advantages
- Can make code **shorter and easier to read** for problems that are naturally self-similar (trees, nested structures, divide-and-conquer algorithms).
- Mirrors mathematical definitions (e.g. factorial, Fibonacci) very directly.
- Well suited to problems with **unknown or variable depth**, such as searching nested directories.

### Disadvantages
- Uses **more memory**, since every call stays on the call stack until it returns.
- Usually **slower** than an equivalent iterative solution, due to function-call overhead.
- Risk of **`RecursionError`** (stack overflow) if the base case is missing or never reached.
- Can be **harder to debug** for beginners, since you must trace multiple nested calls.

---
## Summary

| Concept | Key takeaway |
|---|---|
| Base case | The condition that stops the recursion |
| Recursive case | The call that reduces the problem toward the base case |
| Recurrence relation | A formula defining a problem in terms of smaller versions of itself |
| Recursion vs Iteration | Recursion trades memory/speed for simpler, more elegant code on self-similar problems |

## Practice / Lecture Code
Lecture Code reference: **13.1 – 13.7**

Try it yourself:
1. Write a recursive function to reverse a string.
2. Write a recursive function to compute the sum of a list of numbers.
3. Modify `binary_search` to return **all** indices where the key occurs in a sorted list with duplicates.
4. Rewrite `getDirectorySize` to also return the **number of files** counted, using a helper function.


In [23]:

def factorial_iterative(n):
        if n == 1 or n == 0:
            return 1
        result = 1
        if n < 0:
            raise ValueError("Factorial is not defined for negative numbers.")
        for i in range(2, n + 1):
            result *= i
        return result
factorial_iterative(-5)  # Test the function


ValueError: Factorial is not defined for negative numbers.